# Properati Argentina: Visualizing Geospatial data

In [1]:
# Imports
import csv
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import folium
from sklearn.impute import SimpleImputer


# from statsmodels.stats.outliers_influence import variance_inflation_factor
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## File paths and reproducibility

The original notebook repairs `entrenamiento.csv` because some rows contain more than the expected 25 fields, usually because commas occur inside the description. The repair logic is retained, but now reports and validates the result.

In [2]:
RAW_FILE = Path("entrenamiento.csv")
PARSED_FILE = Path("entrenamiento_parsed.csv")
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
EXPECTED_COLUMNS = 25

In [3]:
def reconstruct_csv(input_file, output_file, expected_columns=25):
    valid = repaired = rejected = 0
    with open(input_file, "r", encoding="utf-8", newline="") as infile, open(output_file, "w", encoding="utf-8", newline="") as outfile:
        reader, writer = csv.reader(infile), csv.writer(outfile)
        header = next(reader)
        writer.writerow(header)
        if len(header) != expected_columns:
            raise ValueError(f"Expected {expected_columns} columns, got {len(header)}")
        for row in reader:
            if len(row) == expected_columns:
                writer.writerow(row); valid += 1
            elif len(row) > expected_columns:
                parsed = row[:21] + [",".join(row[21:-3])] + row[-3:]
                if len(parsed) == expected_columns:
                    writer.writerow(parsed); repaired += 1
                else: rejected += 1
            else: rejected += 1
    return valid, repaired, rejected

if not RAW_FILE.exists():
    raise FileNotFoundError("Place entrenamiento.csv beside this notebook.")
valid, repaired, rejected = reconstruct_csv(RAW_FILE, PARSED_FILE)
print(f"Valid: {valid:,} | Repaired: {repaired:,} | Rejected: {rejected:,}")

Valid: 1,138,951 | Repaired: 2 | Rejected: 4


In [4]:
data = pd.read_csv(PARSED_FILE)
assert len(data.columns) == EXPECTED_COLUMNS
print("Shape:", data.shape)

# data.columns

Shape: (1138953, 25)


## Initial data-quality assessment



In [5]:
# data.head()
columns1 = ['lat', 'lon','l1', 'l2', 'l3', 'rooms', 'bedrooms', 'bathrooms', 
           'surface_total', 'surface_covered', 'currency', 'property_type', 'operation_type','price']

df = data[columns1]

In [6]:
def clean(df):
    # Create a copy to prevent SettingWithCopyWarning
    df_clean = df.copy()

    # Rename geographical columns & label neighborhood
    df_clean = df_clean.rename(columns={"lat": "lon", "lon": "lat"})
    df_clean = df_clean.rename(columns={"l1": "country", "l2": "province", "l3": "neighborhood"})

    # Subset Price data to only consider USD currency
    mask_USD = df_clean["currency"] == "USD"
    # Subset properati data: to only consider homes amongst the properties given 
    mask_home_props = df_clean["property_type"] == "Casa"
    # Subset location data: to only consider Argentina 
    mask_country = df_clean["country"] == "Argentina"

    # Median latitude and longitude by neighborhood
    df_clean["lat"] = df_clean["lat"].fillna(df_clean.groupby("neighborhood")["lat"].transform("median"))
    df_clean["lon"] = df_clean["lon"].fillna(df_clean.groupby("neighborhood")["lon"].transform("median"))
    
    # Fallback to province median
    df_clean["lat"] = df_clean["lat"].fillna(df_clean.groupby("province")["lat"].transform("median"))
    df_clean["lon"] = df_clean["lon"].fillna(df_clean.groupby("province")["lon"].transform("median"))
    
    # Fallback to country median
    df_clean["lat"] = df_clean["lat"].fillna(df_clean.groupby("country")["lat"].transform("median"))
    df_clean["lon"] = df_clean["lon"].fillna(df_clean.groupby("country")["lon"].transform("median"))
    
    # Specify column categories 
    num_cols = ["lat", "lon", "rooms", "bedrooms", "bathrooms", "surface_total", "surface_covered"]
    cat_cols = ["country", "province", "neighborhood", "currency", "property_type", "operation_type"]

     # Fill Numeric columns by → median
    num_cols = df_clean.select_dtypes(include="number").columns
    df_clean[num_cols] = SimpleImputer(strategy="median").fit_transform(df_clean[num_cols])
    
    # Fill Categorical columns by → most frequent value
    cat_cols = df_clean.select_dtypes(exclude="number").columns
    df_clean[cat_cols] = SimpleImputer(strategy="most_frequent").fit_transform(df_clean[cat_cols])

    # Subset data: Remove outliers for "surface_covered"
    low, high = df_clean["surface_covered"].quantile([0.1, 0.9])
    mask_area = df_clean["surface_covered"].between(low, high)
    df_clean = df_clean[mask_area & mask_USD & mask_home_props & mask_country]
    
    return df_clean

In [7]:
df_clean = clean(df)
df_clean.head()

,lon,lat,country,province,neighborhood,rooms,bedrooms,bathrooms,surface_total,surface_covered,currency,property_type,operation_type,price
5658,-58.649,-34.441,Argentina,Bs.As. G.B.A. Zona Norte,Mar del Plata,9.000,2.000,4.000,300.000,75.000,USD,Casa,Venta,"490,000.000"
5661,-58.689,-34.365,Argentina,Bs.As. G.B.A. Zona Norte,Tigre,3.000,2.000,1.000,95.000,75.000,USD,Casa,Venta,"165,000.000"
5662,-58.580,-34.425,Argentina,Bs.As. G.B.A. Zona Norte,Tigre,3.000,2.000,1.000,95.000,75.000,USD,Casa,Venta,"97,000.000"
5666,-65.318,-26.826,Argentina,Tucumán,Yerba Buena,4.000,2.000,3.000,160.000,75.000,USD,Casa,Venta,"160,000.000"
5667,-57.577,-37.986,Argentina,Buenos Aires Costa Atlántica,Mar del Plata,3.000,2.000,1.000,433.000,75.000,USD,Casa,Venta,"105,000.000"


`df_clean` consists of 97, 956homes. In order to reduce computational cost, we will work with the first 200 residences in this dataset in West Zone of Greater Buenos Aires.


In [8]:
# Obtain the first 200 homes in a df_homes dataframe, in West Zone of Greater Buenos Aires
limit = 200
df_homes = df_clean.iloc[0:limit, :]

In [9]:
# West Zone (Zona Oeste) of Greater Buenos Aires latitude and longitude values
latitude = -34.65
longitude = -58.65

In [10]:
# create map and display it
Zona_Oeste_map = folium.Map(location=[latitude, longitude], zoom_start=12)

In [11]:
# instantiate a feature group for the incidents in the dataframe
homes = folium.map.FeatureGroup()

# loop through the 100 crimes and add each to the incidents feature group
for lat, lng, in zip(df_homes.lat, df_homes.lon):
    homes.add_child(
        folium.vector_layers.CircleMarker(
            [lat, lng],
            radius=5, # define how big you want the circle markers to be
            color='yellow',
            fill=True,
            fill_color='blue',
            fill_opacity=0.6
        )
    )

# add incidents to map
Zona_Oeste_map.add_child(homes)

In [12]:
# instantiate a feature group for the incidents in the dataframe
homes = folium.map.FeatureGroup()

# loop through the 100 crimes and add each to the incidents feature group
for lat, lng, in zip(df_homes.lat, df_homes.lon):
    homes.add_child(
        folium.vector_layers.CircleMarker(
            [lat, lng],
            radius=5, # define how big you want the circle markers to be
            color='yellow',
            fill=True,
            fill_color='blue',
            fill_opacity=0.6
        )
    )

# add pop-up text to each marker on the map
latitudes = list(df_homes.lat)
longitudes = list(df_homes.lon)
labels = list(df_homes.operation_type)

for lat, lng, label in zip(latitudes, longitudes, labels):
    folium.Marker([lat, lng], popup=label).add_to(Zona_Oeste_map)    
    
# add incidents to map
Zona_Oeste_map.add_child(homes)

In [13]:
from folium import plugins

# let's start again with a clean copy of the map of Zona Oeste
Zona_Oeste_map = folium.Map(location = [latitude, longitude], zoom_start = 15)

# instantiate a mark cluster object for the homes in the dataframe
homes = plugins.MarkerCluster().add_to(Zona_Oeste_map)

# loop through the dataframe and add each data point to the mark cluster
for lat, lng, label, in zip(df_homes.lat, df_homes.lon, df_homes.operation_type):
    folium.Marker(
        location=[lat, lng], icon=None, popup=label,).add_to(homes)

# display map
Zona_Oeste_map

## Author

<a href="https://www.linkedin.com/in/andrew-kalumba-harris/">ANDREW KALUMBA YIGGA</a><br>
<a href =""> </a>


| Date (YYYY-MM-DD) | Prepared By     | 
| ----------------- | --------------  | 
| 2026-08-22        | Author          | 


## <h3 align="center">  Data Science 2026. <h3/>